# Context-Aware Press Release Distribution Recommender

This notebook implements the thesis proposal as an executable offline research workflow. It maps the proposal objectives to the seven work packages:

| Work package | Notebook implementation |
|---|---|
| WP1 - Data Engineering and Labeling | Load PR exports and outlet engagement report, standardize schema, infer categories, build PR-endpoint interactions, smooth CTR, create temporal split |
| WP2 - Baseline Reproduction | Keyword/rule matching, TF-IDF similarity, static-tier ranking, historical-prior CTR |
| WP3 - Open-Source SLM Encoder Experiments | Distilled and SLM-named compact embedding experiments with local-runtime fallbacks for Llama, Ministral, Phi, Qwen, and SmolLM |
| WP4 - CTR Modeling | Metadata-only logistic predictor and a NumPy DeepFM-style model with calibration |
| WP5 - Graph Authority Module | Outlet graph, PageRank, and hybrid score fusion |
| WP6 - Evaluation and Robustness | Ranking metrics, prediction metrics, ablations, bootstrap confidence interval, limited-data and cold-start slices |
| WP7 - Prototype Integration | Saved artifacts, model card, sample recommendations, and replay/backtesting workflow |

Important data note: the current workspace has PR/post records and outlet-level engagement summaries, but not a true historical PR-to-outlet interaction log. The notebook therefore runs in proxy backtesting mode. When real interaction logs are added, replace the proxy label builder with observed exposures/clicks while keeping the rest of the benchmark protocol.


## Setup

The implementation intentionally uses only `pandas` and `numpy` because this local runtime does not include heavy ML packages. The SLM encoder cells are structured so cached transformer embeddings can be plugged in later, while the current executable path uses deterministic compact hash embeddings as a reproducible fallback.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

from pr_recommender_research_pipeline import ResearchConfig, evaluate_pipeline

ROOT = Path.cwd()
CONFIG = ResearchConfig.from_env()
CONFIG.max_prs = 240
CONFIG.max_outlets = 100
CONFIG.deepfm_epochs = 8
CONFIG.logistic_epochs = 70
CONFIG.parse_xml_bodies = False

ROOT, CONFIG

## WP1 - Data Engineering and Labeling

The pipeline ingests the WordPress PR CSV exports and the expanded outlet distribution report. It standardizes dates, titles, creators, outlet traffic/views/clicks, computes smoothed CTR, infers PR and outlet categories from the provided category taxonomy, constructs a PR-endpoint table, and applies a temporal train/validation/test split.

Because per-PR endpoint exposure logs are absent, `clicked` is created as a proxy relevance/engagement label from available PR-title relevance and outlet-level engagement. This keeps the full research workflow executable while making the data limitation explicit in the exported model card.


In [ ]:
results = evaluate_pipeline(ROOT, CONFIG)

data_profile = results["data_profile"]
pd.Series(data_profile)

## WP2 and WP3 - Baselines and Compact Encoder Experiments

This table compares keyword/rule matching, TF-IDF, a distilled-encoder fallback, and the SLM-named compact encoder experiments. In an environment with local transformer packages and cached models, these score columns are the insertion point for real Llama, Ministral, Phi, Qwen, and SmolLM embeddings.


In [ ]:
encoder_comparison = results["encoder_comparison"]
encoder_comparison[
    ["encoder", "embedding_mode", "precision@10", "recall@10", "map@10", "ndcg@10", "expected_clicks@10"]
].head(12)

## WP4 - CTR Modeling

The supervised engagement layer compares historical endpoint priors, metadata-only logistic regression, and a DeepFM-style model that fuses semantic scores with structured metadata. The DeepFM predictions are Platt-calibrated on the validation split.


In [ ]:
prediction_metrics = results["prediction_metrics"]
prediction_metrics

## WP5 - PageRank Authority and Hybrid Ranking

The graph module builds an outlet graph using shared region/category plus traffic authority priors, runs PageRank, and evaluates hybrid score fusion:

`Score_i = lambda_1 * Rel_i + lambda_2 * CTR_hat_i + lambda_3 * PageRank_i`

Weights are selected on the validation split and reported in the run summary.


In [ ]:
summary = results["summary"]
summary["hybrid_pagerank_weights"], summary["hybrid_no_pagerank_weights"]

In [ ]:
benchmark = results["ranking_metrics"]
benchmark[
    ["ranker", "precision@10", "recall@10", "map@10", "ndcg@10", "expected_clicks@10", "observed_ctr@10"]
].head(15)

## WP6 - Evaluation, Ablation, and Robustness

The benchmark matrix covers ranking metrics. The ablation table isolates encoder, predictor, hybrid ranking, and PageRank contributions. Robustness outputs include a bootstrap confidence interval against static-tier ranking, a low-exposure/cold-start slice when available, and a limited-data study.


In [ ]:
results["ablation_matrix"].head(20)

In [ ]:
results["robustness_checks"]

In [ ]:
results["limited_data_study"]

## WP7 - Prototype Replay Recommendations

The notebook exports a replay recommendation list for one held-out PR using the PageRank-enhanced hybrid ranker. The same scoring path can be reused for backtesting additional held-out PRs.


In [ ]:
recommendations = results["recommendations_sample"]
recommendations.head(20)

## Saved Research Artifacts

The workflow writes the benchmark matrix, ablation matrix, prediction metrics, robustness checks, limited-data study, sample recommendations, data profile, run summary, and model card into `research_outputs/`.


In [ ]:
output_dir = results["output_dir"]
sorted(p.name for p in output_dir.glob("*"))

## Interpretation Checklist

- Use `benchmark_matrix.csv` to determine whether semantic plus CTR plus PageRank improves top-K recommendation quality over static-tier and lexical baselines.
- Use `prediction_metrics.csv` to compare CTR predictors by AUC, log-loss, and calibration error.
- Use `encoder_comparison.csv` to compare lexical, distilled, and SLM encoder variants under the same temporal split.
- Use `ablation_matrix.csv` to isolate the contribution of semantic, CTR, and authority modules.
- Replace proxy labels with true PR-endpoint logs before reporting final thesis claims.
